# Chest X-ray 肺炎分類 — EfficientNet_B3 遷移學習

每個 Cell 對應一個獨立步驟，可分開執行：
- **Cell 1**：環境設定（只需執行一次）
- **Cell 2**：實驗設定（換參數只改這裡，改完重跑 Cell 4+）
- **Cell 3**：下載資料（只需執行一次）
- **Cell 4**：準備資料集與模型
- **Cell 5**：訓練
- **Cell 6**：測試集評估
- **Cell 7**：訓練曲線與混淆矩陣
- **Cell 8**：儲存結果

In [ ]:
import sys
sys.path.insert(0, '..')
# Cell 1 — 環境設定（只需執行一次）
%matplotlib inline
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from constant import DEFAULT_MODEL_NAME, POSITIVE_CLASS_NAME
from data.dataset import build_datasets, download_dataset, make_dataloaders
from engine.trainer import compute_class_weights, run_one_epoch, train_model
from engine.visualize import plot_cm, plot_history
from models.builder import build_model
from utils import EpochTimer, make_output_dir, save_results, set_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用裝置: {device}")

In [ ]:
# Cell 2 — 實驗設定（換參數只改這裡，改完重跑 Cell 4+）
EXPERIMENT_NAME     = "efficientnet_b3"
MODEL_NAME          = DEFAULT_MODEL_NAME
IMAGE_SIZE          = 224
BATCH_SIZE          = 32
EPOCHS              = 12
LR                  = 1e-3
WEIGHT_DECAY        = 1e-4
RANDOM_SEED         = 42

FREEZE_BACKBONE     = True   # 前幾個 epoch 先只訓練分類頭
UNFREEZE_EPOCH      = 4      # 於此 epoch 解凍 backbone
BACKBONE_LR_FACTOR  = 0.1    # 解凍後 backbone 學習率 = LR * factor

set_seed(RANDOM_SEED)

In [ ]:
# Cell 3 — 下載資料（只需執行一次）
dataset_root = download_dataset()
print(f"資料集路徑: {dataset_root}")

In [ ]:
# Cell 4 — 準備資料集與模型
image_datasets, class_to_idx, idx_to_class = build_datasets(dataset_root, img_size=IMAGE_SIZE)
num_classes = len(class_to_idx)
positive_index = class_to_idx.get(POSITIVE_CLASS_NAME, 1)
print(f"類別: {class_to_idx}, 正類別索引({POSITIVE_CLASS_NAME}): {positive_index}")

dataloaders = make_dataloaders(image_datasets, BATCH_SIZE)

model = build_model(
    num_classes, device,
    model_name=MODEL_NAME,
    freeze_backbone=FREEZE_BACKBONE,
)

class_weights = compute_class_weights(image_datasets["train"], num_classes, device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
print(f"模型與資料集準備完成，類別數：{num_classes}")

In [ ]:
# Cell 5 — 訓練
timer = EpochTimer()

best_state, best_val_f1, best_cm, history = train_model(
    model, device, criterion, optimizer, scheduler, dataloaders,
    epochs=EPOCHS,
    unfreeze_epoch=UNFREEZE_EPOCH,
    lr=LR,
    backbone_lr_factor=BACKBONE_LR_FACTOR,
    should_freeze_backbone=FREEZE_BACKBONE,
    positive_index=positive_index,
    epoch_callback=timer,
    early_stopping=True,
    patience=6,
)
print(f"\n訓練完成，最佳驗證 F1：{best_val_f1:.4f}")

time_per_epoch = timer.get_time_per_epoch()

In [ ]:
# Cell 6 — 測試集評估
model.load_state_dict(best_state)
test_m = run_one_epoch(model, device, dataloaders["test"], criterion, positive_index=positive_index)

print(
    "\n[Test] "
    f"loss={test_m['loss']:.4f}, acc={test_m['accuracy']:.4f}, "
    f"precision={test_m['precision']:.4f}, recall={test_m['recall']:.4f}, f1={test_m['f1']:.4f}"
)

In [ ]:
# Cell 7 — 訓練曲線與混淆矩陣
plot_history(history, show=True)
plot_cm(test_m["cm"], class_names=[idx_to_class[0], idx_to_class[1]], show=True)

In [ ]:
# Cell 8 — 儲存結果
out_dir = make_output_dir(EXPERIMENT_NAME, base="../outputs")
save_results(
    out_dir, model, best_state, best_val_f1, test_m,
    class_to_idx, idx_to_class, IMAGE_SIZE, history,
    model_name=MODEL_NAME, train_time_per_epoch=time_per_epoch,
)
print(f"結果已儲存至：{out_dir}")